# Module 087: Building REST APIs with FastAPI

Phase 9: Databases & Web Apps | Duration: 2.5 hours

## Basic FastAPI Setup

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict, List, Optional

app: FastAPI = FastAPI(title='FastAPI Demo')

## Path Operations and Parameters

In [ ]:
@app.get('/')
def root() -> Dict[str, str]:
    return {'message': 'Hello, FastAPI!'}

@app.get('/items/{item_id}')
def read_item(item_id: int, q: Optional[str] = None) -> Dict[str, object]:
    return {'item_id': item_id, 'q': q}

# Test with TestClient
from fastapi.testclient import TestClient
client = TestClient(app)

print(client.get('/').json())
print(client.get('/items/42?q=test').json())

## Request Body with Pydantic Models

In [ ]:
class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True

items_db: List[Item] = []

@app.post('/items/', response_model=Item, status_code=201)
def create_item(item: Item) -> Item:
    items_db.append(item)
    return item

@app.get('/items/', response_model=List[Item])
def list_items() -> List[Item]:
    return items_db

resp = client.post('/items/', json={'name': 'Laptop', 'price': 999.99})
print('Created:', resp.json())
print('All items:', client.get('/items/').json())

## PUT and DELETE

In [ ]:
@app.put('/items/{item_id}', response_model=Item)
def update_item(item_id: int, item: Item) -> Item:
    if item_id >= len(items_db):
        raise HTTPException(status_code=404, detail='Item not found')
    items_db[item_id] = item
    return item

@app.delete('/items/{item_id}')
def delete_item(item_id: int) -> Dict[str, str]:
    if item_id >= len(items_db):
        raise HTTPException(status_code=404, detail='Item not found')
    items_db.pop(item_id)
    return {'message': 'Item deleted'}

print(client.put('/items/0', json={'name': 'Gaming Laptop', 'price': 1499.99}).json())
print(client.delete('/items/0').json())

## Interactive Docs

Run the app with `uvicorn`:

```bash
uvicorn notebook:app --reload
```

Then visit:
- `/docs` — Swagger UI
- `/redoc` — ReDoc alternative

## Dependency Injection Example

In [ ]:
from fastapi import Depends

def common_params(q: Optional[str] = None, skip: int = 0, limit: int = 10):
    return {'q': q, 'skip': skip, 'limit': limit}

@app.get('/products/')
def list_products(params: dict = Depends(common_params)) -> Dict:
    return {'params': params, 'products': ['sample']}

print(client.get('/products/?q=phone&skip=0&limit=5').json())